Idea: 

1. Function create_lstm_model should take in a dict of parameters to define a LSTM Model Architecture
   * this can be very basic but should provide a coverage of options 
2. Function which tunes the hyperparameters
3. run model and log everything using mlflow 

In [1]:
import os
import numpy as np
import pandas as pd 
import itertools
import mlflow

import matplotlib.pyplot as plt

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import explained_variance_score, mean_absolute_error, r2_score, mean_squared_error

# custom functions 
from helper_functions import * 

2023-11-29 20:32:24.255881: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-11-29 20:32:24.255910: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-11-29 20:32:24.255948: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-11-29 20:32:24.262921: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# setting up credentials for storing the model
os.environ["AWS_ACCESS_KEY_ID"] = "eITEO5kyE7hccuy7UTHv"
os.environ["AWS_SECRET_ACCESS_KEY"] = "5KBCscit30Z70bSVGIMvMBBoqV8ydn232o2MW9RA"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.1.0.12:2000"

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [3]:
def create_lstm_model(layers_config):
    model = Sequential()

    for layer_conf in layers_config:
        layer_type = layer_conf["type"]

        if layer_type == "LSTM":
            lstm_kwargs = {
                "units": layer_conf["units"],
                "return_sequences": layer_conf["return_sequences"]
            }
            # Add input_shape only for the first LSTM layer if specified
            if "input_shape" in layer_conf:
                lstm_kwargs["input_shape"] = layer_conf["input_shape"]

            model.add(LSTM(**lstm_kwargs))

        elif layer_type == "Dropout":
            model.add(Dropout(layer_conf["rate"]))

        elif layer_type == "Dense":
            model.add(Dense(units=layer_conf["units"], activation=layer_conf["activation"]))

    return model



In [4]:
fixed_features = ['hour_cos', 'hour_sin', 'day_of_week_cos', 'day_of_week_sin',
                   'month_cos', 'month_sin', 'day_of_month_sin', 'day_of_month_cos',
                     'year_normalized', 'is_holiday', 'volume', 'is_weekend', 'count', 'close_price']
optional_features = ['close_lag12','close_lag6', 'close_lag168', 'sma_6_close_price',
                      'ema_6_close_price', 'ema_12_close_price' ]
#'sma_12_close_price',
# generating all combinations of optional features
all_feature_combinations = []
for r in range(len(optional_features) + 1):
    for subset in itertools.combinations(optional_features, r):
        all_feature_combinations.append(fixed_features + list(subset))

param_grid = {
    'features': all_feature_combinations,
    'lookback': [2, 4, 8, 12, 16],
    #'learning_rate': [0.001, 0.01, 0.1],
    'optimizer': ['adam'],
    'batch_size': [16,32],
    'datatype': [np.float16, np.float32],
    'epochs': [35],
    'loss': ['mean_absolute_error'],
    'metrics' : [['mean_absolute_error', 'mean_squared_error', 'accuracy']] # double list to not alternate between the metrics
}

grid = ParameterGrid(param_grid)

len(grid)
#for i, params in enumerate(grid):
#    # Use params['features'], params['lookback'], etc. to train your model
#    print(params)
#    print(i)
#    pass

1280

In [5]:
def data_prep(data):
    # data prep
    data = data.drop(columns=['open_price', 'high', 'low'], axis=1)

    data = process_timestamp(data, fill=False)
    data = add_feature_date(data)
    data = add_holiday_feature(data)
    data = apply_cyclic_encoding(data, columms=['hour', 'day_of_week', 'month'])
    data = apply_day_of_month_encoding(data)
    data = normalize_year(data) # has to be updated yearly (because of the min-max scaler - max+1 is currently set = 2024)
    data['close_price_true'] =  data['close_price'] # save the true price 
    data = apply_log_scaler(data, columns=['close_price', 'volume', 'count'])
    data = add_feature_lag(data, lags=[1,4,6,12,168])
    data = simple_moving_average(data, window_sizes=[6,12], columns=['close_price'])
    data = exponential_moving_average(data, span_sizes=[6,12], columns=['close_price'])

    # not needed as these values are encoded to be cyclic 
    data = data.drop(columns=['symbol_id', 'hour', 'day_of_week', 'day_of_month', 'month', 'year'], axis=1)

    # dropping nan values which are created because of simple_moving_average and lags 
    data = data.dropna()

    return data

In [6]:
# Prep data once for all models: 
file_path = 'kraken_ohlc_hour_count.csv'
raw_data = pd.read_csv(file_path)
data = data_prep(raw_data)

unseen_test = data.iloc[int(len(data)*0.99):] 
training = data.iloc[:int(len(data)*0.99)]

# Start an MLflow run for each set of parameters
for params in grid:
    with mlflow.start_run():
        # log the currently used parameters
        mlflow.log_params(params)

        # logging training size 
        mlflow.log_param("training data size", len(training))
        mlflow.log_param("unseen test data size", len(unseen_test))

        lookback = params['lookback']
        features = params['features']
            
        # setting up model      
        # default LSTM-Design 
        model_layers_config = [
            {"type": "LSTM", "units": 50, "return_sequences": True, "input_shape": (params['lookback'], len(params['features']))},
            {"type": "Dropout", "rate": 0.2},
            {"type": "LSTM", "units": 50, "return_sequences": False},
            {"type": "Dropout", "rate": 0.2},
            {"type": "Dense", "units": 1, "activation": None}
        ]

        mlflow.log_param("model_layers_config", model_layers_config)

        model = create_lstm_model(model_layers_config)
        # might need Adam(learning_rate=0.001) to set the learning rate 
        model.compile(optimizer=params['optimizer'], loss=params['loss'], metrics=params['metrics'])

        # test if this works
        mlflow.log_param("model summary", model.summary())        
    
        # prep data
        X, y = create_sequences_np(data=training, 
                                   features=params['features'],
                                   lookback=params['lookback'],
                                   datatype=params['datatype'])

        training_validation_split = 0.2
        split_idx = int(len(X) * training_validation_split)

        mlflow.log_param("training_validation_split", training_validation_split)

        # validation set with the last 20 percent of the dataset 
        X_train, X_val = X[:split_idx], X[split_idx:]
        y_train, y_val = y[:split_idx], y[split_idx:]

        # train model
        model.fit(X_train, y_train, epochs=params["epochs"],
                    batch_size=params["batch_size"],
                    validation_data=(X_val, y_val))

        # Evaluate your model
        # Validation Data
        y_pred = model.predict(X_val)

        y_val_true = np.expm1(y_val)
        y_pred_true = np.expm1(y_pred)

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(y_val_true, label='Actual Values')
        ax.plot(y_pred_true, label='Predicted Values')
        ax.set_title('LSTM Model Validation Plot')
        ax.set_xlabel('Time')
        ax.set_ylabel('Close Price')
        ax.set_yscale('log')
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/validation_plot.png')
        plt.close(fig)

        # Unseen Data
        predicted_prices = []
        actual_prices = unseen_test['close_price_true'].values[lookback:]  # actual prices without log transformation
        timestamps = unseen_test['bucket'].values[lookback:]  # corresponding timestamps

        for i in range(lookback, len(unseen_test)):
            last_sequence = unseen_test.iloc[i-lookback:i][features].values.reshape((1, lookback, len(features)))
            last_sequence = np.array(last_sequence).astype(np.float16)
            predicted_log_price = model.predict(last_sequence)
            predicted_price = np.expm1(predicted_log_price)[0, 0]  # inverse log transformation
            predicted_prices.append(predicted_price)
        predicted_prices = np.array(predicted_prices)

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.plot(timestamps, actual_prices, label='Actual Prices', color='blue')
        ax.plot(timestamps, predicted_prices, label='Predicted Prices', color='orange')
        ax.set_title('LSTM Model Predictions vs Actual Prices')
        ax.set_xlabel('Date/Time')
        ax.set_ylabel('Close Price')
        labels = ax.get_xticklabels()
        ax.set_xticklabels(labels, rotation=45)
        ax.legend()
        fig.tight_layout()
        artifact_path = 'plots'
        mlflow.log_figure(fig, artifact_path + '/unseen_test_plot.png')
        plt.close(fig)

        # calculate some metrics 
        mse_value = mean_squared_error(actual_prices, predicted_prices)
        mae_value = mean_absolute_error(actual_prices, predicted_prices)
        rmse_value = np.sqrt(mse_value)
        mape_value = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100
        r2_value = r2_score(actual_prices, predicted_prices)
        explained_variance = explained_variance_score(actual_prices, predicted_prices)

        mlflow.log_metric('mse', mse_value)
        mlflow.log_metric('mae', mae_value)
        mlflow.log_metric('rmse', rmse_value)
        mlflow.log_metric('mape', mape_value)
        mlflow.log_metric('r2', r2_value)
        mlflow.log_metric('explained_variance', explained_variance)

        mlflow.sklearn.log_model(model, "model")


2023-11-29 20:32:28.516173: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-11-29 20:32:28.520983: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-11-29 20:32:28.521142: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 2, 50)             13000     
                                                                 
 dropout (Dropout)           (None, 2, 50)             0         
                                                                 
 lstm_1 (LSTM)               (None, 50)                20200     
                                                                 
 dropout_1 (Dropout)         (None, 50)                0         
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 33251 (129.89 KB)
Trainable params: 33251 (129.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/35


2023-11-29 20:32:32.249670: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2023-11-29 20:32:33.258640: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f2584124290 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-11-29 20:32:33.258665: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1060 6GB, Compute Capability 6.1
2023-11-29 20:32:33.262490: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-11-29 20:32:33.335935: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


839/839 [==============================] - 14s 13ms/step - loss: 0.1433 - mean_absolute_error: 0.1433 - mean_squared_error: 0.0604 - accuracy: 0.1193 - val_loss: 0.0679 - val_mean_absolute_error: 0.0679 - val_mean_squared_error: 0.0081 - val_accuracy: 0.1250
Epoch 2/35
839/839 [==============================] - 10s 12ms/step - loss: 0.0923 - mean_absolute_error: 0.0923 - mean_squared_error: 0.0336 - accuracy: 0.1213 - val_loss: 0.0468 - val_mean_absolute_error: 0.0468 - val_mean_squared_error: 0.0038 - val_accuracy: 0.1250
Epoch 3/35
839/839 [==============================] - 10s 12ms/step - loss: 0.0871 - mean_absolute_error: 0.0871 - mean_squared_error: 0.0312 - accuracy: 0.1216 - val_loss: 0.0641 - val_mean_absolute_error: 0.0641 - val_mean_squared_error: 0.0065 - val_accuracy: 0.1250
Epoch 4/35
839/839 [==============================] - 9s 11ms/step - loss: 0.0844 - mean_absolute_error: 0.0844 - mean_squared_error: 0.0291 - accuracy: 0.1217 - val_loss: 0.0409 - val_mean_absolute_er

/tmp/ipykernel_98418/2049789421.py:110: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels(labels, rotation=45)
/home/rileydavid/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 4, 50)             13000     
                                                                 
 dropout_2 (Dropout)         (None, 4, 50)             0         
                                                                 
 lstm_3 (LSTM)               (None, 50)                20200     
                                                                 
 dropout_3 (Dropout)         (None, 50)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 51        
                                                                 
Total params: 33251 (129.89 KB)
Trainable params: 33251 (129.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/35
839/8

KeyboardInterrupt: 